# 03 · Visualising the comparison

**Mission covered:** 6 ✅

**Question:** how can we show at a glance what notebook 02 demonstrated with numbers?

**What we learn:** the structure of a matplotlib figure (`fig`, `ax`), choosing the right chart,
exporting images for the README.

**Rules of a good chart:** a title, labelled axes **with their unit**, a legend, distinct colours.

## Setup

These cells are the same as in `01_exploration.ipynb`: imports, FastF1 cache and the `show` function.
The detailed explanations are in notebook 01.

> 🔁 Copying this block into every notebook is not ideal: if it is fixed in one place, it has to be fixed everywhere.
> This is exactly the problem that **mission 8** will solve (moving this code into `src/`).

In [ ]:
import os

import fastf1
import fastf1.plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# The relative path "../cache" is only correct if the notebook runs from notebooks/
assert os.path.basename(os.getcwd()) == "notebooks", f"Unexpected working directory: {os.getcwd()}"

cache_dir = os.path.abspath("../cache")
os.makedirs(cache_dir, exist_ok=True)
print("FastF1 cache:", cache_dir)

fastf1.Cache.configure(cache_dir=cache_dir)

In [ ]:
def show(df, columns=None, n=None):
    '''Display a DataFrame as a Jupyter table, without truncated columns.

    columns : list of column names to keep (all by default)
    n       : number of rows to display (all by default)
    '''
    if columns is not None:
        df = df[columns]
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None):
        display(df)

### Loading the session


Same race as in `02_driver_comparison.ipynb`.

In [ ]:
YEAR = 2024
GRAND_PRIX = "Monaco"
SESSION_TYPE = "R"

session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION_TYPE)
session.load()
print(session.event["EventName"], "-", session.name)

In [ ]:
# Without a Formula 1 account sign-in, FastF1 only retrieves the results:
# accessing session.laps then raises a DataNotLoadedError.
try:
    laps_count = len(session.laps)
except fastf1.exceptions.DataNotLoadedError:
    laps_count = 0

if laps_count == 0:
    print("⚠️ No laps loaded. Sign in from a terminal with:")
    print("   python -m fastf1 auth f1tv --authenticate")
else:
    print(f"✅ {laps_count} laps loaded")

## 6.1 · Getting the clean laps

Same filtering and cleaning as in notebook 02.
(This is copied code: it will be factored out in mission 8.)

In [ ]:
DRIVER_1 = "LEC"
DRIVER_2 = "PIA"
DRIVERS = [DRIVER_1, DRIVER_2]

In [ ]:
laps = session.laps.pick_drivers(DRIVERS)

clean_laps = (
    laps
    .pick_wo_box()            # remove pit out-laps and in-laps
    .pick_track_status("1")   # keep only laps fully under green flag
    .pick_accurate()          # keep lap times FastF1 considers reliable
    .pick_not_deleted()       # remove laps deleted by race control (track limits)
)
clean_laps = clean_laps[clean_laps["LapNumber"] > 1]   # lap 1 (standing start) is not representative

# Lap time in seconds: a float is easier to work with than a duration
clean_laps = clean_laps.assign(LapTime_s=clean_laps["LapTime"].dt.total_seconds())

print("Representative laps:", clean_laps["Driver"].value_counts().to_dict())

## 6.2 · Preparing the export

Images go into `image/` at the root of the repo. Same logic as for the cache: from `notebooks/`,
the relative path is `../image`.

In [ ]:
image_dir = os.path.abspath("../image")
os.makedirs(image_dir, exist_ok=True)
print("Images:", image_dir)

## 6.3 · Chart 1: lap time evolution

- X axis: lap number. Y axis: time in seconds.
- One point per representative lap, one colour per driver.
- `fastf1.plotting.get_driver_color` gives the official colour of the driver's team.

Structure of a matplotlib figure: `fig` is the whole figure (the image), `ax` is the plotting area (the axes).
We draw on `ax`, and export `fig`.

In [ ]:
fig_laps, ax = plt.subplots(figsize=(10, 5))

for driver in DRIVERS:
    driver_laps = clean_laps[clean_laps["Driver"] == driver]
    colour = fastf1.plotting.get_driver_color(driver, session)
    ax.plot(driver_laps["LapNumber"], driver_laps["LapTime_s"],
            marker="o", markersize=4, linewidth=1, color=colour, label=driver)

ax.set_xlabel("Lap")
ax.set_ylabel("Lap time (s)")
ax.set_title(f"{session.event['EventName']} {YEAR}: lap times of {DRIVER_1} and {DRIVER_2}")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 6.4 · Chart 2: lap time distribution

A **boxplot** summarises the distribution of each driver's lap times:
- the middle line is the **median**;
- the box contains half of the laps (from the 25th to the 75th percentile): the smaller it is, the more **consistent** the driver;
- isolated points are unusual laps.

In [ ]:
fig_distribution, ax = plt.subplots(figsize=(6, 5))

data = [clean_laps.loc[clean_laps["Driver"] == driver, "LapTime_s"] for driver in DRIVERS]
boxes = ax.boxplot(data, tick_labels=DRIVERS, patch_artist=True, medianprops={"color": "black"})

for box, driver in zip(boxes["boxes"], DRIVERS):
    box.set_facecolor(fastf1.plotting.get_driver_color(driver, session))
    box.set_alpha(0.6)

ax.set_ylabel("Lap time (s)")
ax.set_title("Distribution of representative lap times")
ax.grid(axis="y", alpha=0.3)
plt.show()

## 6.5 · Chart 3: stints by tyre compound

One chart per driver, side by side (`plt.subplots(1, 2)`), with the same Y axis (`sharey=True`) so they can be compared.
Each point is coloured by the tyre used, with the official colours (`get_compound_color`).
The hard tyre (`HARD`) is white: a black outline (`edgecolors`) keeps the points visible.

In [ ]:
fig_tyres, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, driver in zip(axes, DRIVERS):
    driver_laps = clean_laps[clean_laps["Driver"] == driver]
    for compound, compound_laps in driver_laps.groupby("Compound"):
        ax.scatter(compound_laps["LapNumber"], compound_laps["LapTime_s"], s=20,
                   color=fastf1.plotting.get_compound_color(compound, session),
                   edgecolors="black", linewidths=0.5, label=compound)
    ax.set_title(driver)
    ax.set_xlabel("Lap")
    ax.legend(title="Tyre")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Lap time (s)")
fig_tyres.suptitle("Lap times by tyre compound")
plt.show()

## 6.6 · Exporting

`dpi=150` gives a sharp image, `bbox_inches="tight"` removes unnecessary white margins.

In [ ]:
figures = {
    "lap_times.png": fig_laps,
    "lap_time_distribution.png": fig_distribution,
    "lap_times_by_tyre.png": fig_tyres,
}

for name, figure in figures.items():
    path = os.path.join(image_dir, name)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    print("Exported:", path)

## What the charts show

- **Lap times**: the two curves overlap almost all the way. In Monaco, overtaking is nearly impossible:
  Piastri stays behind Leclerc, and both manage their tyres by driving slowly early in the race
  (84 s on lap 3, 78.5 s around lap 20). Lap times then drop steadily as the car burns fuel.
- **End of the race**: between laps 66 and 72, Leclerc clearly speeds up (down to 75.2 s) while Piastri stays
  around 77 s. This is where most of the final gap is built. Leclerc's last lap, slower (78.6 s),
  is a celebration lap.
- **Distribution**: the medians are at the same level (about 78.5 s). Leclerc's box reaches lower
  because of his fast laps at the end of the race.
- **Tyres**: both drivers ran **the whole race on hard tyres**. They started on mediums but
  changed tyres during the first-lap red flag, which spared them a pit stop.
  This chart will be more interesting on a race with several stints (try changing `GRAND_PRIX`).